In [0]:
from pyspark.sql.functions import col, concat, lit, expr, count, sum as spark_sum, extract, countDistinct, min as spark_min, lower, substring, explode, sequence, to_date

members = spark.read.table("members")
facilities = spark.read.table("facilities")
bookings = spark.read.table("bookings")

f = facilities.alias("f")
b = bookings.alias("b")
m = members.alias("m")

## JOIN

---

#### 1. easy - https://pgexercises.com/questions/joins/simplejoin.html
##### *How can you produce a list of the start times for bookings by members named 'David Farrell'?*


In [0]:
spark.sql("""
        select starttime from bookings
        left join members
        on bookings.memid = members.memid
        where members.firstname = 'David' and members.surname = 'Farrell';
        """).display()


starttime
2012-09-18T09:00:00.000Z
2012-09-18T17:30:00.000Z
2012-09-18T13:30:00.000Z
2012-09-18T20:00:00.000Z
2012-09-19T09:30:00.000Z
2012-09-19T15:00:00.000Z
2012-09-19T12:00:00.000Z
2012-09-20T15:30:00.000Z
2012-09-20T11:30:00.000Z
2012-09-20T14:00:00.000Z


---
####  2. easy - https://pgexercises.com/questions/joins/simplejoin2.html
##### *How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.*

In [0]:
spark.sql("""
          select starttime, fac.name from bookings
          left join facilities fac
          on bookings.facid = fac.facid
          where fac.name like 'Tennis Court%'
          and starttime like '2012-09-21%'
          order by starttime
          """).display()

starttime,name
2012-09-21T08:00:00.000Z,Tennis Court 2
2012-09-21T08:00:00.000Z,Tennis Court 1
2012-09-21T09:30:00.000Z,Tennis Court 1
2012-09-21T10:00:00.000Z,Tennis Court 2
2012-09-21T11:30:00.000Z,Tennis Court 2
2012-09-21T12:00:00.000Z,Tennis Court 1
2012-09-21T13:30:00.000Z,Tennis Court 1
2012-09-21T14:00:00.000Z,Tennis Court 2
2012-09-21T15:30:00.000Z,Tennis Court 1
2012-09-21T16:00:00.000Z,Tennis Court 2


---
#### 3. easy - https://pgexercises.com/questions/joins/self2.html
##### *How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname).*

In [0]:
members = spark.read.table("members")

m1 = members.alias("m1")
m2 = members.alias("m2")

joined = m1.join(m2, m1.recommendedby == m2.memid, "left").orderBy("m1.surname", "m1.firstname")
result = joined.select(col("m1.surname").alias("surname"), col("m1.firstname").alias("firstname"), col("m2.surname").alias("referred_surname"), col("m2.firstname").alias("referred_firstname"))
result.display()

surname,firstname,referred_surname,referred_firstname
Bader,Florence,Stibbons,Ponder
Baker,Anne,Stibbons,Ponder
Baker,Timothy,Farrell,Jemima
Boothe,Tim,Rownam,Tim
Butters,Gerald,Smith,Darren
Coplin,Joan,Baker,Timothy
Crumpet,Erica,Smith,Tracy
Dare,Nancy,Joplette,Janice
Farrell,David,null,null
Farrell,Jemima,null,null


---
#### 4. medium - https://pgexercises.com/questions/joins/threejoin.html (three join)
##### *How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.*


In [0]:
joined = b.join(m, b.memid == m.memid, "left").join(f, b.facid == f.facid, "left")
result = joined.select(concat(col("m.firstname"), lit(" "), col("m.surname")).alias("member"), col("f.name").alias("facility")).distinct().where(col("f.name").like("Tennis Court%")).orderBy("member", "facility")
result.display()

member,facility
Anne Baker,Tennis Court 1
Anne Baker,Tennis Court 2
Burton Tracy,Tennis Court 1
Burton Tracy,Tennis Court 2
Charles Owen,Tennis Court 1
Charles Owen,Tennis Court 2
Darren Smith,Tennis Court 2
David Farrell,Tennis Court 1
David Farrell,Tennis Court 2
David Jones,Tennis Court 1


---
#### 5. medium - https://pgexercises.com/questions/joins/sub.html (subquery and join)
##### *How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered.*

In [0]:
spark.sql("""
            select distinct concat(firstname, ' ', surname) as member, (
                select STRING_AGG(concat(firstname, ' ', surname)) as recommender from members rec
                where mem.recommendedby = rec.memid
            ) from members mem
            order by member;
            """).display()

# Correlated scalar subqueries aren’t fully supported in Spark SQL (unless aggregated), hence the use of STRING_AGG.

member,"(SELECTSTRING_AGG(concat(firstname,' ',surname))ASrecommenderFROMmembersrecWHEREmem.recommendedby=rec.memid)"
Anna Mackenzie,Darren Smith
Anne Baker,Ponder Stibbons
Burton Tracy,null
Charles Owen,Darren Smith
Darren Smith,null
David Farrell,null
David Jones,Janice Joplette
David Pinker,Jemima Farrell
Douglas Jones,David Jones
Erica Crumpet,Tracy Smith



## AGGREGATION

---
#### 1. easy - https://pgexercises.com/questions/aggregates/count3.html Group by order by
##### *Produce a count of the number of recommendations each member has made. Order by member ID.*

In [0]:
m = members.alias("m")
result = m.groupBy(col("recommendedby")).agg(count("*")).orderBy("recommendedby").dropna()
result.display()


recommendedby,count(1)
1,5
2,3
3,1
4,2
5,1
6,1
9,2
11,1
13,2
15,1


---
#### 2. easy - https://pgexercises.com/questions/aggregates/fachours.html group by order by
##### *Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.*

In [0]:
result = b.groupBy("facid").agg(spark_sum("slots")).orderBy("facid")
result.display()

facid,sum(slots)
0,1320
1,1278
2,1209
3,830
4,1404
5,228
6,1104
7,908
8,911


---
#### 3. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition
##### *Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.*

In [0]:
result = b.where(col("b.starttime").like("2012-09-%")).groupBy("facid").agg(spark_sum("slots").alias("Total slots")).orderBy("Total slots")
result.display()

facid,Total slots
5,122
3,422
7,426
8,471
6,540
2,570
1,588
0,591
4,648


---
#### 4. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
##### *Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month.*

In [0]:
result = b.where(col("starttime").like("2012-%")).groupBy("facid", extract(lit("month"), "starttime").alias("month")).agg(spark_sum("slots").alias("Total slots")).orderBy("facid", "month")
result.display()

facid,month,Total slots
0,7,270
0,8,459
0,9,591
1,7,207
1,8,483
1,9,588
2,7,180
2,8,459
2,9,570
3,7,104


---
#### 5. easy - https://pgexercises.com/questions/aggregates/members1.html count distinct
##### *Find the total number of members (including guests) who have made at least one booking.*

In [0]:
result = b.agg(countDistinct("memid").alias("count"))
result.display()

count
30


---
#### 6. med - https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join
##### *Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID.*

In [0]:
joined = b.join(m, b.memid == m.memid, "left")
result = joined.groupBy("m.surname", "m.firstname", "b.memid").agg(spark_min("b.starttime").alias("starttime")).orderBy("memid")
result.display()

surname,firstname,memid,starttime
GUEST,GUEST,0,2012-07-03T18:00:00.000Z
Smith,Darren,1,2012-07-03T08:00:00.000Z
Smith,Tracy,2,2012-07-04T09:00:00.000Z
Rownam,Tim,3,2012-07-04T13:30:00.000Z
Joplette,Janice,4,2012-07-10T08:30:00.000Z
Butters,Gerald,5,2012-07-16T11:00:00.000Z
Tracy,Burton,6,2012-07-26T09:00:00.000Z
Dare,Nancy,7,2012-07-26T09:30:00.000Z
Boothe,Tim,8,2012-07-26T08:00:00.000Z
Stibbons,Ponder,9,2012-08-04T09:30:00.000Z



## STRING AND DATE

---
#### 1. easy - https://pgexercises.com/questions/string/concat.html format string
##### *Output the names of all members, formatted as 'Surname, Firstname'*

In [0]:
result = m.select(concat(col("surname"), lit(", "), col("firstname")))
result.display()

"concat(surname, , , firstname)"
"GUEST, GUEST"
"Smith, Darren"
"Smith, Tracy"
"Rownam, Tim"
"Joplette, Janice"
"Butters, Gerald"
"Tracy, Burton"
"Dare, Nancy"
"Boothe, Tim"
"Stibbons, Ponder"


---
#### 2. easy - https://pgexercises.com/questions/string/case.html WHERE + string function
##### *Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns.*

In [0]:
result = f.select(col("name")).where(lower(col("name")).like("tennis%"))
result.display()

name
Tennis Court 1
Tennis Court 2


---
#### 3. easy - https://pgexercises.com/questions/string/reg.html WHERE + string function
##### *You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.*

In [0]:
result = m.select(col("memid"), col("telephone")).where(col("telephone").like("(%)%"))
result.display()


memid,telephone
0,(000) 000-0000
3,(844) 693-0723
4,(833) 942-4710
5,(844) 078-4130
6,(822) 354-9973
7,(833) 776-4001
8,(811) 433-2547
9,(833) 160-3900
10,(855) 542-5251
11,(844) 536-8036


---
#### 4. easy - https://pgexercises.com/questions/string/substr.html group by, substr
##### *You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0.*

In [0]:
result = m.groupBy(substring("surname", 1, 1).alias("letter")).agg(count("*")).orderBy("letter")
result.display()

letter,count(1)
B,5
C,2
D,1
F,2
G,2
H,1
J,3
M,1
O,1
P,2


---
#### 5. easy - https://pgexercises.com/questions/date/series.html generate ts
##### *Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.*

In [0]:
df = spark.createDataFrame([""])
result = df.select(explode(sequence(to_date(lit("2012-10-01")), to_date(lit("2012-10-31")))).alias("ts"))
result.display()

ts
2012-10-01
2012-10-02
2012-10-03
2012-10-04
2012-10-05
2012-10-06
2012-10-07
2012-10-08
2012-10-09
2012-10-10


---
#### 6. easy - https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts
##### *Return a count of bookings for each month, sorted by month*

In [0]:
result = b.groupBy(extract(lit("month"), "starttime").alias("month")).agg(count("bookid")).orderBy("month")
result.display()

month,count(bookid)
1,1
7,658
8,1472
9,1913


## Question
##### *How can you produce a list of the start times for bookings by members named 'David Farrell'?*

In [0]:
joined = b.join(m, b.memid == m.memid, "left")
result = joined.select(col("b.starttime")).where(col("m.surname").like("Farrell") & col("m.firstname").like("David"))
result.display()

starttime
2012-09-18T09:00:00.000Z
2012-09-18T17:30:00.000Z
2012-09-18T13:30:00.000Z
2012-09-18T20:00:00.000Z
2012-09-19T09:30:00.000Z
2012-09-19T15:00:00.000Z
2012-09-19T12:00:00.000Z
2012-09-20T15:30:00.000Z
2012-09-20T11:30:00.000Z
2012-09-20T14:00:00.000Z
